In [1]:
import os, numpy as np, pandas as pd

# ===== CONFIG =====
IN_DIR  = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files"
OUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files"

INSTRUMENT = "RX1"  # change to AD1 etc.
EWMA_FILE  = f"{INSTRUMENT}_EWMA_4d_16d.csv"   # choose which EWMA cross
CARRY_FILE = f"{INSTRUMENT}_CARRY.csv"
OUT_FILE   = f"{INSTRUMENT}_COMBINED.csv"

WEIGHT_EWMA  = 0.5
WEIGHT_CARRY = 0.5
CAP = 20.0
TRADING_DAYS = 256

# ===== LOAD =====
ewma = pd.read_csv(os.path.join(IN_DIR, EWMA_FILE))
carry = pd.read_csv(os.path.join(IN_DIR, CARRY_FILE))

for df in (ewma, carry):
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

# merge by Date
df = pd.merge(ewma, carry, on="Date", suffixes=("_ewma", "_carry")).sort_values("Date")

# choose price column
price_col = "PX_CLOSE_1D_ewma" if "PX_CLOSE_1D_ewma" in df else "PX_CLOSE_1D"
px = df[price_col].astype(float)
ret = px.pct_change()

# ===== compute raw PnL per strategy =====
pnl_ewma  = df["forecast_ewma"].shift(1) * ret
pnl_carry = df["forecast_carry"].shift(1) * ret

# drop missing values for correlation
mat = pd.concat([pnl_ewma, pnl_carry], axis=1).dropna()
if len(mat) > 10:
    C = mat.corr().values
    w = np.array([WEIGHT_EWMA, WEIGHT_CARRY]).reshape(-1, 1)
    fdm = 1.0 / np.sqrt(float(w.T @ C @ w))
    fdm = min(fdm, 2.5)
else:
    fdm = 1.0

print(f"{INSTRUMENT} Forecast Diversification Multiplier (FDM): {fdm:.3f}")

# ===== combine forecasts =====
f_comb = (WEIGHT_EWMA * df["forecast_ewma"] + WEIGHT_CARRY * df["forecast_carry"]) * fdm
f_comb = f_comb.clip(-CAP, CAP)
df["forecast_combined"] = f_comb

# ===== compute raw Sharpe of combined layer =====
raw_pnl_comb = f_comb.shift(1) * ret
mu, sd = raw_pnl_comb.mean(), raw_pnl_comb.std()
sharpe_comb = np.nan if sd == 0 or np.isnan(sd) else (mu / sd) * np.sqrt(TRADING_DAYS)

print(f"{INSTRUMENT} Combined Raw Sharpe: {sharpe_comb:.3f}")

# ===== SAVE =====
df_out = df[["Date", "forecast_carry", "forecast_ewma", "forecast_combined"]]
df_out.to_csv(os.path.join(OUT_DIR, OUT_FILE), index=False)
print(f"Saved combined forecast: {os.path.join(OUT_DIR, OUT_FILE)}")


C:\Users\loci_\AppData\Local\Temp\ipykernel_18780\1762641851.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fdm = 1.0 / np.sqrt(float(w.T @ C @ w))


RX1 Forecast Diversification Multiplier (FDM): 1.353
RX1 Combined Raw Sharpe: 0.015
Saved combined forecast: C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files\RX1_COMBINED.csv
